# Grid search de hiperparámetros — BETO + LoRA

In [ ]:
# TODO = AGREGAR SIN HASHTAGS
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
import pandas as pd
from itertools import product
from transformers import AutoTokenizer

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed, save_metrics
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_search
from betohumor.plots import plot_training_curves, plot_grid_search_comparison

/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Datos y configuraciones

In [2]:
data_config = DataConfig(data_path = "../data/raw/haha_2019_train.csv")
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)
tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)

Train: 19197 | Val: 2397 | Test: 2400


## 2. Definición de la grilla
`builder` recibe los params de cada combinación y arma un `LoraConfig`
propio para esa corrida — `beto_config` queda fijo (cerrado por la función).

In [ ]:
R_VALUES = [8, 16, 32]
LR_VALUES = [1e-4, 5e-4, 1e-3, 5e-3]
WEIGHT_DECAYS = [0.01, 0.05]

search_grid = [
    {
        'r':             r,
        'lora_alpha':    r * 2,
        'learning_rate': lr,
        'weight_decay':  wd,
    }
    for r, lr, wd in product(R_VALUES, LR_VALUES, WEIGHT_DECAYS)
]

search_grid

Total combinaciones: 24


[{'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0005, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.0005, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.001, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.001, 'weight_decay': 0.05},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.005, 'weight_decay': 0.01},
 {'r': 8, 'lora_alpha': 16, 'learning_rate': 0.005, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0005, 'weight_decay': 0.01},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.0005, 'weight_decay': 0.05},
 {'r': 16, 'lora_alpha': 32, 'learning_rate': 0.001, 'weight_decay': 0.01},
 {'r': 16, '

## 3. Correr el search

In [4]:
results = run_search(
    lambda params: build_beto_lora(beto_config, LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])), search_grid, beto_config, data_config,
    df_train, df_val, tokenizer, seed=data_config.seed,
    output_dir_prefix='results/checkpoints/search_lora',
)

print('\n=== RESULTADOS (ordenados por Macro F1) ===')
for r in results:
    print(f"  {r['params']} -> Macro F1: {r['macro_f1']:.4f}")


=== r16_lora_alpha32_learning_rate0.0001 ===


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 68287.43it/s]
BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECT

trainable params: 591,362 || all params: 110,443,780 || trainable%: 0.5354


/opt/miniconda3/envs/AP_FINAL/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

## 4. Curvas de entrenamiento de cada combinación

In [ ]:
for r in results:
    print(f"\n--- {r['run_name']} (Macro F1: {r['macro_f1']:.4f}) ---")
    plot_training_curves(r['history'])

## 5. Comparación entre configuraciones

In [ ]:
labels = [r['run_name'] for r in results]
scores = [r['macro_f1'] for r in results]
plot_grid_search_comparison(labels, scores, title='Grid search Macro F1 en validación')

## 6. Guardar resultados en CSV


In [ ]:
all_metrics = {
    r['run_name']: {'macro_f1': r['macro_f1'], 'params': r['params']}
    for r in results
}

save_metrics(all_metrics, folder_name='hyperparams_search_lora')